In [1]:
from elasticsearch7 import Elasticsearch
from elasticsearch7.client import IndicesClient
import json
import string
import os

from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

In [2]:
FINAL_INDEX_FOLDER='./final_index'

## Stopwords

In [3]:
swPath = "./stoplist.txt"

with open(swPath) as file:
    stopwords = file.readlines()
    
    for index, stopword in enumerate(stopwords):
        stopwords[index] = stopword.split("\n")[0]
        

# Adding punctuations in the stopwords list
punctuations = list(string.punctuation)

# Extra
extraPunc = ["``", "'s'", "'", "''"]
[punctuations.append(el) for el in extraPunc]

for p in punctuations:
    stopwords.append(p)
        
print(f'Total number of stopwords: {len(stopwords)}')

Total number of stopwords: 531


## Elastic Search

In [4]:
cloud_id='1b6862a6400b430ca00ffa767c181e4a:dXMtY2VudHJhbDEuZ2NwLmNsb3VkLmVzLmlvOjQ0MyRhMmQxM2IwMTA0N2E0OTlkOWNhMWVjZDYzODQwZDMwZiQwNjc4YzkxMGNlMjM0MTRhYjg5MjJjZTBlZDE4NjFjZA==' 
es=Elasticsearch(request_timeout=10000, cloud_id=cloud_id, http_auth=('elastic', 'uF0a5ShswLkhZmAfPnIffttD'))

In [5]:
NUM_DOCS=0
LENGTH_OF_ALL_DOCS = 0
 
def create_index():
    index_json = {
        "settings": {
            "number_of_shards": 1,
            "number_of_replicas": 1,
            "max_result_window": 100000,
            "analysis": {
                "filter": {
                    "english_stop": {
                        "type": "stop",
                        "stopwords": stopwords
                    }
                },
                "analyzer": {
                    "stopped": {
                        "type": "custom",
                        "tokenizer": "standard",
                        "filter": [
                            "lowercase",
                            "english_stop"
                        ]
                    }
                }
            }
        },
        "mappings": {
            "properties": {
                "content": {
                    "type": "text",
                    "fielddata": True,
                    "analyzer": "stopped",
                    "index_options": "positions"

                },
                "url": {
                    "type": "text"
                },
                "inlinks": {
                    "type": "text"
                },
                "outlinks": {
                    "type": "text"
                },
                "author":{
                    "type":"text"
                },
                "title":{
                    "type":"text"
                }
            }
        }
    }
    es.indices.create(index="hw3",body=index_json)

In [6]:
def merge(_id,inlinks,url,outlinks,title,body,author):
    ret={
        "id":_id,
        "url": url,
        "title":title,
        "content":body,
        "inlinks":inlinks,
        "outlinks":outlinks,
        "author":author
    }
    
    script = {
        "script": {
            "source": """
                for (item in params.inlinks) {
                    if (!ctx._source['inlinks'].contains(item)) {
                        ctx._source['inlinks'].add(item);
                    }
                } 
                for (item in params.outlinks) {
                    if (!ctx._source['outlinks'].contains(item)) {
                        ctx._source['outlinks'].add(item);
                    }
                }
                if (!ctx._source['author'].contains(params.author[0])) {
                    ctx._source['author'].add(params.author[0]);
                }
            """,
            "params": ret
        }
    }
    
    if es.exists(index="hw3",id=_id):
        es.update(index="hw3",id=_id,body=script)
    else:
        insert_body={
            "title":title,
            "content":body,
            "url": url,
            "inlinks":inlinks,
            "outlinks":outlinks,
            "author":author
        }
        es.index(index="hw3",body=insert_body,id=_id)

In [7]:
files = os.listdir(FINAL_INDEX_FOLDER)

print(f'Total files: {len(files) - 1}')

for index, file in enumerate(files):
    if "json" not in file:
        continue
        
    print(f'On index: {index}')
    
    with open(FINAL_INDEX_FOLDER + '/' + file) as curr_file:
        links_data=json.load(curr_file)
        for url in links_data.keys():
            curr_data=links_data[url]
            _id = f'{url[:400]}_{hash(url)}'
            merge(_id,curr_data['inlinks'],url,curr_data['outlinks'],curr_data['title'],curr_data['content'],curr_data['author'])

Total files: 59
On index: 0


C:\Users\sanje\AppData\Local\Temp\ipykernel_36088\589414945.py:44: DeprecationWarning: The 'body' parameter is deprecated for the 'index' API and will be removed in a future version. Instead use the 'document' parameter. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  es.index(index="hw3",body=insert_body,id=_id)


On index: 1
On index: 2
On index: 3
On index: 4
On index: 5
On index: 6
On index: 7
On index: 8
On index: 9
On index: 10
On index: 11
On index: 12
On index: 13
On index: 14
On index: 15
On index: 16
On index: 17
On index: 18
On index: 19
On index: 20
On index: 21
On index: 22
On index: 23
On index: 24
On index: 25
On index: 26
On index: 27
On index: 28
On index: 29
On index: 30
On index: 31
On index: 32
On index: 33
On index: 34
On index: 35
On index: 36
On index: 37
On index: 38
On index: 39
On index: 40
On index: 41
On index: 42
On index: 43
On index: 44
On index: 45
On index: 46
On index: 47
On index: 48
On index: 49
On index: 50
On index: 51
On index: 52
On index: 53
On index: 54
On index: 55
On index: 56
On index: 57
On index: 58
On index: 59
